<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2017%20-%20Neural%20Networks%20II/lab8_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 8: Neural Networks - Predicting car prices @ Toyota**

Consider the data on used cars (_ToyotaCorolla.csv_) with 1436 records and details on 38 attributes, including Price, Age, KM, HP, and other specifications. The goal is to predict the price of a used Toyota Corolla based on its specifications.

Last session you traced a forward pass through a small network by hand. Today scikit-learn does that arithmetic, and the training, for us. One difference from the slides: the worked examples there were **classification** (like / dislike). Here the output is a **number** (a price), so we use `MLPRegressor`, the regression version of the same network.

**Learning goals**
- Prepare data for a neural network: turn categories into dummy columns and scale every column to the 0–1 range.
- Fit a neural network to a regression problem with `MLPRegressor`, and convert its predictions back to euros.
- Compare network architectures against a linear-regression baseline using validation RMSE.
- Recognize a network that failed to train, and fix it.

## Setup

In [ ]:
# Install the textbook's helper package (needed in Colab), then import everything the lab uses.
!pip install -q dmba
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
from dmba import regressionSummary

## Part 1: Preparing the data

In [ ]:
# Load the used-car data and look at its size and first rows.
car_df = pd.read_csv('https://raw.githubusercontent.com/AnalyticsArmory/data/main/ToyotaCorolla.csv')
print(car_df.shape)
car_df.head()

Use Price as the outcome and the following variables as predictors: Age_08_04, KM, Fuel_Type, HP, Automatic, Doors, Quarterly_Tax, Mfr_Guarantee, Guarantee_Period, Airco, Automatic_airco, CD_Player, Powered_Windows, Sport_Model, and Tow_Bar

### The variables we use

Each row is one used Corolla offered for sale by a Toyota dealer in the Netherlands, so prices and taxes are in **euros**.

| Column | What it means | Values in this data |
|---|---|---|
| `Price` | Offer price in euros, **the outcome we predict** | €4,350 – €32,500 |
| `Age_08_04` | Age of the car in months, as of August 2004 | 1 – 80 |
| `KM` | Kilometers on the odometer | 1 – 243,000 |
| `Fuel_Type` | Fuel: Petrol, Diesel or CNG (compressed natural gas) | 3 categories |
| `HP` | Horsepower | 69 – 192 |
| `Automatic` | Automatic gearbox (1) or manual (0) | 0 / 1 |
| `Doors` | Number of doors | 2 – 5 |
| `Quarterly_Tax` | Road tax paid every three months, in euros | €19 – €234 |
| `Mfr_Guarantee` | Still covered by the manufacturer's guarantee (1) or not (0) | 0 / 1 |
| `Guarantee_Period` | Length of the guarantee, in months | 3 – 36 |
| `Airco` | Has air conditioning | 0 / 1 |
| `Automatic_airco` | Has automatic (climate-control) air conditioning | 0 / 1 |
| `CD_Player` | Has a CD player | 0 / 1 |
| `Powered_Windows` | Has electric windows | 0 / 1 |
| `Sport_Model` | Is the sport version of the model | 0 / 1 |
| `Tow_Bar` | Has a tow bar | 0 / 1 |

### The columns we leave out, and why

The file has 39 columns; we use 16. The other 23 fall into four groups:

- **Labels, not measurements.** `Id` is just a row number. `Model` is the full model name, with 319 different values: as dummy columns it would add 318 inputs, most of them describing only a handful of cars.
- **Repeats of Age.** `Mfg_Year` and `Mfg_Month` are the manufacturing date, which is what `Age_08_04` already measures (the correlation between `Mfg_Year` and age is −0.98). Including them would give the network the same information twice.
- **Almost the same for every car.** `Cylinders` is 4 for every car, and `Parking_Assistant`, `Power_Steering`, `Gears` and `Airbag_1` take one value for 97% or more of the cars. A column that barely varies says almost nothing about how one car differs from another, so it probably would not help much in telling a cheap car from an expensive one.
- **Left out to keep the network small.** `Weight`, `CC` (engine size), `Color`, `Met_Color`, `BOVAG_Guarantee`, `ABS`, `Airbag_2`, `Boardcomputer`, `Central_Lock`, `Radio`, `Radio_cassette`, `Mistlamps`, `Backseat_Divider` and `Metallic_Rim` are real car features and could carry information about price. They are left out only to keep this first network small: every extra input adds one more weight to fit for each node in the first hidden layer. This follows the textbook's version of the exercise (DMBA Ch. 11). Exercise 3 at the end asks you to put one back.

In [ ]:
# Keep Price and the 15 predictors.
selected_var = ['Price', 'Age_08_04', 'KM', 'Fuel_Type', 'HP', 'Automatic', 'Doors',
                'Quarterly_Tax', 'Mfr_Guarantee', 'Guarantee_Period', 'Airco',
                'Automatic_airco', 'CD_Player', 'Powered_Windows', 'Sport_Model', 'Tow_Bar']
car_df = car_df[selected_var]

Before going further, check what kind of values each column holds. A neural network only does arithmetic, so every input has to be a number.

In [ ]:
# Show each column's data type and how many distinct values it has.
pd.DataFrame({'type': car_df.dtypes, 'distinct values': car_df.nunique()})

`Fuel_Type` is the only text column, with three values: CNG, Diesel and Petrol. We replace it with 0/1 **dummy columns**. `drop_first=True` leaves out the first category (CNG), because a car with 0 in both remaining columns must be CNG.

In [ ]:
# Replace Fuel_Type with 0/1 dummy columns.
car_df = pd.get_dummies(car_df, columns=['Fuel_Type'], drop_first=True)
car_df.head()

### Scaling every column to 0–1

A network adds up weighted inputs, so a column measured in big numbers drowns out the rest: `KM` runs into the hundreds of thousands while `Doors` is 2 to 5. **MinMax scaling** puts every column on the same 0–1 range:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

The smallest value becomes 0 and the largest becomes 1.

**Split first, then scale.** The validation cars stand in for future cars the model has never seen, so they must not influence any part of training, and that includes the scaling. We therefore learn each column's min and max from the **training** cars only (`fit_transform`), then scale the validation cars with those same numbers (`transform`). As a result a few validation values can land slightly below 0 or above 1, which is fine.

We scale the outcome, `Price`, as well. A new network starts with small random weights, so its first predictions are close to 0, not close to €10,000. Training goes faster and more reliably when the target is on a similar small scale. The catch: the network's predictions then come out on the 0–1 scale too, and we have to convert them back to euros before judging them.

In [ ]:
# Separate the outcome (Price) from the predictors.
y_nonscaled = car_df[['Price']]
X_nonscaled = car_df.drop(columns=['Price'])
X_nonscaled.head()

In [ ]:
# Hold out 30% of the cars for validation.
X_train_nonscaled, X_valid_nonscaled, y_train_nonscaled, y_valid_nonscaled = train_test_split(
    X_nonscaled, y_nonscaled, test_size=0.3, random_state=1)

In [ ]:
# Scale to 0-1 using the min and max of the training cars only (one scaler for the predictors, one for Price).
scaleInput = MinMaxScaler()
scaleOutput = MinMaxScaler()
X_train = scaleInput.fit_transform(X_train_nonscaled)  # learn min and max from the training cars, then scale them
X_valid = scaleInput.transform(X_valid_nonscaled)      # scale the validation cars with those same min and max

# .ravel() flattens the one-column result into a plain list, which fit() expects
y_train = scaleOutput.fit_transform(y_train_nonscaled).ravel()
y_valid = scaleOutput.transform(y_valid_nonscaled).ravel()

In [ ]:
# Check the scaling: training values run exactly from 0 to 1, validation values roughly so.
print('training:   smallest', X_train.min(), '  largest', X_train.max())
print('validation: smallest', X_valid.min(), '  largest', X_valid.max().round(3))

## Part 2: A first neural network

### One hidden layer with 2 nodes

`MLPRegressor` builds and trains the network ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)). The settings we use:

- `hidden_layer_sizes` — one number per hidden layer, giving how many nodes it has. `(2,)` is one hidden layer with 2 nodes; `(5, 5, 5)` would be three hidden layers of 5 nodes each. (The comma in `(2,)` is how Python writes a list with a single item.)
- `activation='logistic'` — the S-shaped sigmoid function from the slides.
- `solver='lbfgs'` — the method that adjusts the weights during training. It works well on small datasets like this one.
- `random_state=1` — fixes the random starting weights, so everyone gets the same result.

In [ ]:
# Train a network with one hidden layer of 2 nodes.
car_nnet = MLPRegressor(hidden_layer_sizes=(2,), activation='logistic', solver='lbfgs', random_state=1)
car_nnet.fit(X_train, y_train)

In [ ]:
# Predict the validation prices and look at their range.
y_pred_scaled = car_nnet.predict(X_valid)
print('min predicted price:', y_pred_scaled.min(), '  max predicted price:', y_pred_scaled.max())

All the predictions are between 0 and 1, because we scaled Price. To read them as prices we "denormalize" them with the scaler's `inverse_transform`, which runs the MinMax formula backwards. We will do this for every model, so it gets a small helper function.

In [ ]:
# Helper: undo the 0-1 scaling so prices are back in euros.
def to_euros(scaled_prices):
    # inverse_transform expects a one-column table, so reshape the list into a column first
    return scaleOutput.inverse_transform(scaled_prices.reshape(-1, 1)).ravel()

In [ ]:
# Convert the predictions and the actual validation prices back to euros.
y_pred = to_euros(y_pred_scaled)
y_actual = to_euros(y_valid)
print('min predicted price:', round(y_pred.min()), '  max predicted price:', round(y_pred.max()))

### How good is it?

`regressionSummary(actual, predicted)`, from the textbook's `dmba` package, reports the validation errors in euros. It does not include **R²**, so we add it in a small helper.

- **RMSE** is roughly how far off a typical prediction is, in euros. Lower is better, and it is the number we compare models on.
- **R²** is the share of the variation in prices that the model explains. 1 would be perfect; 0 means no better than predicting the average price for every car.

In [ ]:
# Helper: the textbook's error report, plus R-squared on the last line.
def regression_report(actual, predicted):
    regressionSummary(actual, predicted)
    print(f'{"R-squared (R2)":>37} : {r2_score(actual, predicted):.4f}')

In [ ]:
# Report the validation errors, in euros.
regression_report(y_actual, y_pred)

In [ ]:
# Plot predicted against actual price: a perfect model would put every car on a straight diagonal line.
pd.DataFrame({'actual': y_actual, 'predicted': y_pred}).plot.scatter(x='actual', y='predicted', alpha=0.5)

### Compare to linear regression

Is the network worth the trouble? Fit a plain linear regression on exactly the same scaled data and the same split, and score it the same way.

In [ ]:
# Fit a linear regression on the same training data and score it the same way.
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = to_euros(lr_model.predict(X_valid))
regression_report(y_actual, y_pred)

## Part 3: Changing the architecture

### One hidden layer with 5 nodes

In [ ]:
# Train and score a network with one hidden layer of 5 nodes.
car_nnet = MLPRegressor(hidden_layer_sizes=(5,), activation='logistic', solver='lbfgs', random_state=1)
car_nnet.fit(X_train, y_train)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

### Deep learning: 3 hidden layers, each with 2 nodes

In [ ]:
# Train and score a network with three hidden layers of 2 nodes each.
car_nnet = MLPRegressor(hidden_layer_sizes=(2, 2, 2), activation='logistic', solver='lbfgs', random_state=1)
car_nnet.fit(X_train, y_train)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

### Deep learning: 3 hidden layers, each with 5 nodes

In [ ]:
# Train and score a network with three hidden layers of 5 nodes each.
car_nnet = MLPRegressor(hidden_layer_sizes=(5, 5, 5), activation='logistic', solver='lbfgs', random_state=1)
car_nnet.fit(X_train, y_train)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

### Diagnosing a network that failed to train

Total failure: the RMSE is about three times worse than every other model. A useful yardstick is the "model" that ignores the car completely and predicts the average training price for every car. Any model worth using must beat it.

In [ ]:
# Yardstick: the error from predicting the average training price for every car.
average_price = to_euros(y_train).mean()
print('average price:', round(average_price))
regression_report(y_actual, [average_price] * len(y_actual))

Almost exactly the same error, and an R² of about 0 for both: predicting the average is exactly what an R² of 0 means, and the network does no better. It has learned next to nothing. Two quick checks show why:

In [ ]:
# Check the range of its predictions and how many training steps it took.
print('min predicted price:', round(y_pred.min()), '  max predicted price:', round(y_pred.max()))
print('training steps taken:', car_nnet.n_iter_)

Every car gets nearly the same price, and training stopped after only a handful of steps.

Training stops for one of two reasons: it reaches the step limit `max_iter` (200 by default), or a step improves the fit by less than `tol`, its threshold for "no longer worth continuing". A handful of steps is far below 200, so the limit was not the problem. The solver quit early because its first steps barely improved anything. A deeper network starts out further from a good answer, and with three layers of small random weights the signal that tells it which way to move is very faint at the start, so the early steps genuinely look useless and the solver gives up.

The natural first guess, "give it more iterations", therefore does not help on its own:

In [ ]:
# First guess: allow up to 1000 training steps instead of 200.
car_nnet = MLPRegressor(hidden_layer_sizes=(5, 5, 5), activation='logistic', solver='lbfgs', max_iter=1000, random_state=1)
car_nnet.fit(X_train, y_train)
print('training steps taken:', car_nnet.n_iter_)
regression_report(y_actual, to_euros(car_nnet.predict(X_valid)))

Same number of steps, same error: a higher limit changes nothing when training never reaches it.

**The fix: tell the solver not to quit so early.** Lowering `tol` to `1e-8` (0.00000001) means even tiny improvements count as worth continuing. Raise `max_iter` as well, so it has room to finish.

In [ ]:
# Fix: lower tol so training keeps going, and allow up to 2000 steps.
car_nnet = MLPRegressor(hidden_layer_sizes=(5, 5, 5), activation='logistic', solver='lbfgs',
                        tol=1e-8, max_iter=2000, random_state=1)
car_nnet.fit(X_train, y_train)
print('training steps taken:', car_nnet.n_iter_)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

Now it trains for well over a thousand steps and the error falls from about €3,571 to about €1,198. The network has learned. But it is no better than the linear regression, and worse than the tiny one-layer network with 2 nodes: three layers of 5 nodes bought nothing here.

**What about a different activation function?** ReLU, `max(0, x)`, is the usual choice in deep networks. Try it with the default settings:

In [ ]:
# Same architecture with the ReLU activation and default training settings.
car_nnet = MLPRegressor(hidden_layer_sizes=(5, 5, 5), activation='relu', solver='lbfgs', random_state=1)
car_nnet.fit(X_train, y_train)
print('training steps taken:', car_nnet.n_iter_)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

Stuck again: an error near the yardstick after only 19 steps. Changing the activation function on its own is not a reliable fix. Apply the same fix as before:

In [ ]:
# ReLU plus the same fix: lower tol and allow more steps.
car_nnet = MLPRegressor(hidden_layer_sizes=(5, 5, 5), activation='relu', solver='lbfgs',
                        tol=1e-8, max_iter=2000, random_state=1)
car_nnet.fit(X_train, y_train)
print('training steps taken:', car_nnet.n_iter_)
y_pred = to_euros(car_nnet.predict(X_valid))
regression_report(y_actual, y_pred)

### Summary

| Model | Validation RMSE | Validation R² |
|---|---|---|
| Predict the average price for every car (yardstick) | €3,560 | 0.00 |
| Linear regression | €1,193 | 0.89 |
| NN (2), logistic | €1,022 | 0.92 |
| NN (5), logistic | €1,057 | 0.91 |
| NN (2, 2, 2), logistic | €1,099 | 0.90 |
| NN (5, 5, 5), logistic, default settings | €3,571 | −0.01 |
| NN (5, 5, 5), logistic, `tol=1e-8`, `max_iter=2000` | €1,198 | 0.89 |
| NN (5, 5, 5), relu, default settings | €3,562 | 0.00 |
| NN (5, 5, 5), relu, `tol=1e-8`, `max_iter=2000` | €1,033 | 0.92 |

*(Numbers from scikit-learn 1.9. Other versions can differ by a few euros.)*

Overall, the neural network architecture has only a small effect on performance here: the small networks all land within about €80 of each other and beat linear regression by roughly €100–170. Going deeper did not help: the best network is still the smallest one. The deeper networks are also where training can fail outright, because there are many more weights to fit. There we have to be more diligent: compare against a simple yardstick, check how many steps training took, and adjust the settings when it quit early.

Fine-tuning neural networks, especially in a deep learning context, is not trivial... recommend taking a deep learning class!

## Follow up Questions and Exercises

**Question 1.** The best networks beat linear regression, but only by a little. Give one reason a manager might still choose the linear regression.

**Answer**

*Provide your answer here*

**Question 2.** Why is "predict the average price for every car" a quick test of whether a model learned anything? What did it tell us about the (5, 5, 5) logistic network?

**Answer**

*Provide your answer here*

**Question 3.** What would go wrong if you computed the RMSE on the scaled 0–1 predictions instead of converting them back to euros first?

**Answer**

*Provide your answer here*

**Exercise 1.** Choose your own architecture, for example `(10,)`, `(10, 10)` or `(3, 3)`. Train it with `activation='logistic'` and score it as above. Do more layers or more nodes help?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 2.** Re-run the original (5, 5, 5) logistic network (default `tol` and `max_iter`) with `random_state` set to 2, 3, 4, 5 and 6, changing nothing else. Check the RMSE and `n_iter_` each time. What do you notice?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

**Exercise 3.** Add `Weight` back as a predictor: put it in `selected_var` and re-run the notebook from that cell down. Compare the RMSE of the linear regression and of the (2,) network with what you had before. Did both improve?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*